In [1]:
import sys, os, sqlite3, json, time, re
import pandas as pd
import numpy as np
import platform, warnings
from tqdm import tqdm

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
tqdm.pandas()

sys.path.append('..')
from config import DB_PATH

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)
print(f'DB 연결: {db_path}')

DB 연결: ../data/dave_diver.db


In [2]:
import sqlite3
import os
import pandas as pd

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)

# 모든 테이블 조회
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(f"📋 테이블 목록 ({len(tables)}개)")
print(tables.to_string(index=False))

# 각 테이블별 컬럼 정보 + 행 수
for table in tables['name']:
    print(f"\n{'='*50}")
    print(f"📊 {table}")
    print(f"{'='*50}")
    
    # 컬럼 정보
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    print(schema[['cid', 'name', 'type', 'notnull', 'dflt_value']].to_string(index=False))
    
    # 행 수
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    print(f"\n총 행 수: {count['cnt'][0]:,}행")

conn.close()

📋 테이블 목록 (12개)
                   name
     cleaned_reviews_en
     cleaned_reviews_ja
     cleaned_reviews_ko
   cooccurrence_results
 neg_reviews_classified
 pos_reviews_classified
                reviews
     sentiment_keywords
sentiment_model_summary
   topic_assignments_en
   topic_assignments_ko
      topic_definitions

📊 cleaned_reviews_en
 cid                        name    type  notnull dflt_value
   0                   review_id    TEXT        0       None
   1                    voted_up INTEGER        0       None
   2                review_month    TEXT        0       None
   3                play_segment    TEXT        0       None
   4          playtime_at_review INTEGER        0       None
   5              playtime_hours    REAL        0       None
   6           received_for_free INTEGER        0       None
   7 written_during_early_access INTEGER        0       None
   8                 review_text    TEXT        0       None
   9                cleaned_text    TEXT 

---
## 2. 일본어 리뷰 로드 & 기초 통계

In [19]:
df_ja = pd.read_sql("""
    SELECT *, playtime_at_review / 60.0 as playtime_hours
    FROM reviews
    WHERE language = 'japanese'
""", conn)

print(f'일본어 리뷰: {len(df_ja):,}건')
print(f'긍정: {df_ja["voted_up"].sum():,}건 ({df_ja["voted_up"].mean()*100:.1f}%)')
print(f'부정: {(~df_ja["voted_up"].astype(bool)).sum():,}건')

def segment_label(h):
    if h < 2: return 'casual(<2h)'
    elif h < 10: return 'regular(2-10h)'
    elif h < 30: return 'engaged(10-30h)'
    elif h < 50: return 'engaged(30-50h)'
    else: return 'hardcore(50h+)'

df_ja['play_segment'] = df_ja['playtime_hours'].apply(segment_label)
segment_order = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']

print(f'\n구간별 긍정률:')
for seg in segment_order:
    subset = df_ja[df_ja['play_segment'] == seg]
    if len(subset) == 0: continue
    rate = subset['voted_up'].mean() * 100
    print(f'  {seg:20s}: {len(subset):>4}건, 긍정률 {rate:.1f}%')

일본어 리뷰: 1,023건
긍정: 938건 (91.7%)
부정: 85건

구간별 긍정률:
  casual(<2h)         :   31건, 긍정률 74.2%
  regular(2-10h)      :  168건, 긍정률 91.1%
  engaged(10-30h)     :  358건, 긍정률 89.9%
  engaged(30-50h)     :  255건, 긍정률 92.9%
  hardcore(50h+)      :  211건, 긍정률 96.2%


---
## 3. 일본어 텍스트 전처리 (spaCy ja)

spaCy의 일본어 모델(ja_core_news_sm)은 SudachiPy 기반.
형태소 분석 → 명사/동사/형용사만 추출 → 불용어 제거.

In [20]:
import spacy

nlp_ja = spacy.load('ja_core_news_sm')

JA_STOPWORDS = set([
    'の', 'に', 'は', 'を', 'た', 'が', 'で', 'て', 'と', 'し', 'れ', 'さ',
    'ある', 'いる', 'する', 'なる', 'ない', 'この', 'それ', 'こと', 'もの',
    'ところ', 'ため', 'よう', 'ほう', 'たち', 'など', 'まま', 'そう',
    'です', 'ます', 'から', 'けど', 'でも', 'だけ', 'まで', 'より',
])


def clean_japanese(text: str) -> str:
    """일본어 리뷰 전처리"""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ''
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)  # wwwww → ww

    doc = nlp_ja(text)
    tokens = []
    for token in doc:
        if token.pos_ in ('NOUN', 'VERB', 'ADJ'):
            lemma = token.lemma_
            if lemma not in JA_STOPWORDS and len(lemma) > 1:
                tokens.append(lemma)
    return ' '.join(tokens)


test = 'このゲームは面白いけどボスが難しすぎる'
print(f'원본: {test}')
print(f'전처리: {clean_japanese(test)}')
print(f'\nspaCy ja 로드 완료')

원본: このゲームは面白いけどボスが難しすぎる
전처리: ゲーム 面白い ボス 難しい すぎる

spaCy ja 로드 완료


In [21]:
print('일본어 전처리 중...')
df_ja['cleaned_text'] = df_ja['review_text'].progress_apply(clean_japanese)

empty = (df_ja['cleaned_text'] == '') | df_ja['cleaned_text'].isna()
print(f'전처리 완료: {len(df_ja):,}건, 빈 텍스트: {empty.sum()}건')
df_ja = df_ja[~empty].copy()
df_ja['review_month'] = pd.to_datetime(df_ja['timestamp_created'], unit='s').dt.strftime('%Y-%m')

일본어 전처리 중...


100%|██████████| 1023/1023 [00:25<00:00, 39.96it/s]

전처리 완료: 1,023건, 빈 텍스트: 29건


In [22]:
ja_cols = ['review_id', 'voted_up', 'review_month', 'play_segment',
           'playtime_at_review', 'playtime_hours',
           'received_for_free', 'written_during_early_access',
           'review_text', 'cleaned_text']

df_ja[ja_cols].to_sql('cleaned_reviews_ja', conn, if_exists='replace', index=False)
conn.commit()
print(f'cleaned_reviews_ja 저장: {len(df_ja):,}건')

cleaned_reviews_ja 저장: 994건


---
## 4. 일본어 부정 리뷰 LLM 분류 (Sonnet)

In [23]:
df_ja_neg = df_ja[df_ja['voted_up'] == 0].copy()
df_ja_neg['has_dev_response'] = 0
df_ja_neg['text_len'] = df_ja_neg['review_text'].str.len()

print(f'일본어 부정 리뷰: {len(df_ja_neg):,}건')
for seg in segment_order:
    n = len(df_ja_neg[df_ja_neg['play_segment'] == seg])
    if n > 0: print(f'  {seg}: {n}건')

일본어 부정 리뷰: 85건
  casual(<2h): 8건
  regular(2-10h): 15건
  engaged(10-30h): 36건
  engaged(30-50h): 18건
  hardcore(50h+): 8건


In [24]:
CATEGORIES = ['gameplay', 'story_content', 'repetition', 'technical', 'company', 'forced_neg', 'other']
VALID_CATEGORIES = set(CATEGORIES)
VALID_TONES = {'constructive', 'emotional', 'mixed', 'troll'}
VALID_SENTIMENTS = {'pure_negative', 'mixed_negative', 'positive_but_negative_vote', 'joke'}


def build_ja_prompt(reviews_batch):
    reviews_text = ''
    for item in reviews_batch:
        text = item['text'][:500].replace('"', "'")
        reviews_text += f'[{item["idx"]}] {text}\n\n'

    return f"""You are an expert game review analyst classifying negative Steam reviews of "Dave the Diver".
Reviews are in Japanese. Consider Japanese gaming culture and internet slang.

## Japanese slang guide
- クソゲー (kusoge) = bad game
- 神ゲー (kamige) = masterpiece
- 作業ゲー (sagyo-ge) = grindy/repetitive game
- ヌルゲー (nuru-ge) = too easy
- 課金 (kakin) = microtransaction
- ネクソン/NEXON = publisher (often negative in JP)
- つまらない = boring
- 飽きる/飽きた = got bored
- バグ = bug, ラグ = lag

## Categories
gameplay: Controls, combat, boss fights, minigames, difficulty, balance.
story_content: Story, characters, ending, dialogue.
repetition: Repetitive loop, lack of content. Key: 作業ゲー, 飽きた.
technical: Bugs, crashes, performance.
company: Nexon, pricing, DLC. Key: 課金, ネクソン.
forced_neg: Not genuine. Protest, joke.
other: Doesn't fit above.

## Tone: constructive / emotional / mixed / troll
## Actual sentiment: pure_negative / mixed_negative / positive_but_negative_vote / joke

## Reviews
{reviews_text}

## Output
JSON array. Each: "idx", "category", "sub_category" (or "none"), "tone", "actual_sentiment", "confidence", "reason" (English).
Return {len(reviews_batch)} items. JSON only, no markdown."""


def classify_ja_batch(reviews_batch, model='claude-sonnet-4-5-20250929'):
    import anthropic
    client = anthropic.Anthropic()
    prompt = build_ja_prompt(reviews_batch)
    response = client.messages.create(
        model=model, max_tokens=4096,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return parse_ja_response(response.content[0].text, len(reviews_batch))


def parse_ja_response(text, expected):
    default = {'category': 'other', 'sub_category': 'none', 'tone': 'mixed',
               'actual_sentiment': 'pure_negative', 'confidence': 'low', 'reason': 'parse_failed'}

    def validate(item):
        return {
            'category': item.get('category', 'other') if item.get('category') in VALID_CATEGORIES else 'other',
            'sub_category': item.get('sub_category', 'none') if (item.get('sub_category') in VALID_CATEGORIES or item.get('sub_category') == 'none') else 'none',
            'tone': item.get('tone', 'mixed') if item.get('tone') in VALID_TONES else 'mixed',
            'actual_sentiment': item.get('actual_sentiment', 'pure_negative') if item.get('actual_sentiment') in VALID_SENTIMENTS else 'pure_negative',
            'confidence': item.get('confidence', 'unknown'),
            'reason': str(item.get('reason', ''))[:200],
        }

    cleaned = re.sub(r'^```(?:json)?\s*', '', text.strip())
    cleaned = re.sub(r'\s*```$', '', cleaned)

    # 1차: 전체 JSON
    try:
        results = json.loads(cleaned)
        if isinstance(results, list):
            v = [validate(i) for i in results]
            while len(v) < expected: v.append({**default, 'reason': 'missing'})
            return v[:expected]
    except json.JSONDecodeError: pass

    # 2차: 배열 부분 추출
    m = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if m:
        try:
            results = json.loads(m.group())
            if isinstance(results, list):
                v = [validate(i) for i in results]
                while len(v) < expected: v.append(dict(default))
                return v[:expected]
        except: pass

    return [dict(default) for _ in range(expected)]


print('일본어 분류 함수 정의 완료')

일본어 분류 함수 정의 완료


In [27]:
JA_TABLE = 'neg_reviews_classified'
BATCH_SIZE = 20

# 이미 분류된 리뷰 확인
try:
    already = set(pd.read_sql(f"""
        SELECT review_id FROM {JA_TABLE}
        WHERE classify_method = 'anthropic' AND language_group = 'japanese'
    """, conn)['review_id'])
except: already = set()

remaining = df_ja_neg[~df_ja_neg['review_id'].isin(already)].copy()
print(f'분류 대상: {len(remaining):,}건 (이미 완료: {len(already):,}건)')

if len(remaining) > 0:
    total_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in tqdm(range(0, len(remaining), BATCH_SIZE),
                      total=total_batches, desc='japanese (sonnet)'):
        batch_df = remaining.iloc[start:start+BATCH_SIZE]
        batch_input = [{'idx': j+1, 'text': row['review_text']}
                       for j, (_, row) in enumerate(batch_df.iterrows())]
        try:
            results = classify_ja_batch(batch_input)
            rows = []
            for i, (_, row) in enumerate(batch_df.iterrows()):
                r = results[i] if i < len(results) else {
                    'category': 'other', 'sub_category': 'none',
                    'tone': 'mixed', 'actual_sentiment': 'pure_negative',
                    'confidence': 'low', 'reason': 'error'}
                rows.append({
                    'review_id': row['review_id'], 'classify_method': 'anthropic',
                    'language_group': 'japanese',
                    'voted_up': row['voted_up'], 'review_month': row.get('review_month'),
                    'play_segment': row['play_segment'],
                    'playtime_at_review': row.get('playtime_at_review'),
                    'playtime_hours': row.get('playtime_hours'),
                    'received_for_free': row.get('received_for_free'),
                    'written_during_early_access': row.get('written_during_early_access'),
                    'language': 'japanese',
                    'votes_up': row.get('votes_up'), 'votes_funny': row.get('votes_funny'),
                    'weighted_vote_score': row.get('weighted_vote_score'),
                    'steam_purchase': row.get('steam_purchase'),
                    'has_dev_response': 0,
                    'num_games_owned': row.get('num_games_owned'),
                    'num_reviews': row.get('num_reviews'),
                    'text_len': len(str(row.get('review_text', ''))),
                    'review_text': row['review_text'],
                    'cleaned_text': row['cleaned_text'],
                    'category': r['category'], 'sub_category': r.get('sub_category', 'none'),
                    'tone': r.get('tone', 'mixed'),
                    'actual_sentiment': r.get('actual_sentiment', 'pure_negative'),
                    'confidence': r['confidence'], 'reason': r['reason'],
                })
            pd.DataFrame(rows).to_sql(JA_TABLE, conn, if_exists='append', index=False)
            conn.commit()
        except Exception as e:
            print(f'\n  [에러] {e}')
        time.sleep(1.0)

print(f'\nDB 현황:')
print(pd.read_sql(f"""
    SELECT language_group, COUNT(*) as n
    FROM {JA_TABLE} WHERE classify_method = 'anthropic'
    GROUP BY language_group
""", conn).to_string(index=False))

분류 대상: 85건 (이미 완료: 0건)


japanese (sonnet): 100%|██████████| 5/5 [03:03<00:00, 36.76s/it]


DB 현황:
language_group    n
       english 1649
      japanese   85
        korean  298


---
## 5. 가설 3 검증: 일본어 vs 영어/한국어 비교

In [28]:
cls_all = pd.read_sql(f"""
    SELECT * FROM {JA_TABLE}
    WHERE classify_method = 'anthropic'
""", conn)

print('=== 3개 언어 부정 카테고리 비교 ===')
lang_cat = pd.crosstab(cls_all['language_group'], cls_all['category'], normalize='index') * 100
lang_cat = lang_cat.reindex(columns=CATEGORIES).fillna(0).round(1)
print(lang_cat.to_string())

if 'japanese' in lang_cat.index and 'english' in lang_cat.index:
    diff = lang_cat.loc['japanese'] - lang_cat.loc['english']
    print(f'\n--- 일본어 - 영어 차이 (pp) ---')
    for cat in CATEGORIES:
        if abs(diff[cat]) >= 3:
            direction = '일본어↑' if diff[cat] > 0 else '영어↑'
            print(f'  {cat:15s}: {diff[cat]:+.1f}pp ({direction})')

=== 3개 언어 부정 카테고리 비교 ===
category        gameplay  story_content  repetition  technical  company  forced_neg  other
language_group                                                                            
english             30.7           10.0        25.7       10.2      9.6         2.5   11.1
japanese            48.2            1.2        35.3       10.6      1.2         0.0    3.5
korean              26.2           13.1        22.5       15.1      6.4         2.3   14.4

--- 일본어 - 영어 차이 (pp) ---
  gameplay       : +17.5pp (일본어↑)
  story_content  : -8.8pp (영어↑)
  repetition     : +9.6pp (일본어↑)
  company        : -8.4pp (영어↑)
  other          : -7.6pp (영어↑)


In [29]:
print('=== 3개 언어 tone 비교 ===')
lang_tone = pd.crosstab(cls_all['language_group'], cls_all['tone'], normalize='index') * 100
print(lang_tone.round(1).to_string())

print(f'\n=== 3개 언어 actual_sentiment 비교 ===')
lang_sent = pd.crosstab(cls_all['language_group'], cls_all['actual_sentiment'], normalize='index') * 100
print(lang_sent.round(1).to_string())

=== 3개 언어 tone 비교 ===
tone            constructive  emotional  mixed  troll
language_group                                       
english                 63.2       22.4   11.5    2.9
japanese                42.4       38.8   16.5    2.4
korean                  44.0       41.9   10.4    3.7

=== 3개 언어 actual_sentiment 비교 ===
actual_sentiment  joke  mixed_negative  positive_but_negative_vote  pure_negative
language_group                                                                   
english            2.4            50.9                         4.9           41.8
japanese           0.0            47.1                         1.2           51.8
korean             2.7            40.3                         2.7           54.4


In [30]:
cats_plot = [c for c in CATEGORIES if c != 'other']
lang_labels = {'english': '영어', 'korean': '한국어', 'japanese': '일본어'}
lang_colors = {'english': '#45B7D1', 'korean': '#4ECDC4', 'japanese': '#FF6B6B'}

fig = go.Figure()
for lang in ['english', 'korean', 'japanese']:
    if lang not in lang_cat.index: continue
    n = len(cls_all[cls_all['language_group'] == lang])
    fig.add_trace(go.Scatterpolar(
        r=[lang_cat.loc[lang, c] for c in cats_plot],
        theta=cats_plot, fill='toself',
        name=f'{lang_labels[lang]} (n={n:,})', opacity=0.6,
        line=dict(color=lang_colors[lang])))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 50])),
    title='가설3: 3개 언어 부정 리뷰 프로필 비교',
    height=550, width=650, template='plotly_white')
fig.show()

In [31]:
cls_ja = cls_all[cls_all['language_group'] == 'japanese'].copy()

print(f'=== 일본어 부정 리뷰 카테고리별 샘플 ===')
for cat in CATEGORIES:
    subset = cls_ja[cls_ja['category'] == cat]
    if len(subset) == 0: continue
    print(f'\n--- {cat} ({len(subset)}건) ---')
    for _, row in subset.head(3).iterrows():
        print(f'  [{row["play_segment"]}] [{row["tone"]}] {row["review_text"][:200]}')
        if row.get('reason'): print(f'    → {row["reason"]}')

=== 일본어 부정 리뷰 카테고리별 샘플 ===

--- gameplay (41건) ---
  [regular(2-10h)] [emotional] ボス戦勝てない
クリオネがひどい
相手は体当たりでゴリゴリ酸素を削ってくるのにこっちは銃を構えていてもそれを解除されてなにもできない
ボス戦だけおもんない
    → Complains about boss fight mechanics, specifically the Clione boss being unfair - player gets oxygen depleted while unable to attack properly. Considers boss battles the only unfun part.
  [engaged(30-50h)] [emotional] くそげー。楽しいの最初だけ。そのうち深海に潜って荷物満載で移動速度が落ちたところでサメに襲われて死んで、１個しかアイテムもって帰れない。どんだけ手間と時間かけてると思ってんだ。このゲームをSteamのライブラリから削除します。
    → Calls it kusoge, criticizes death penalty system where you can only bring back 1 item after investing time and effort. States they will delete the game from Steam library.
  [engaged(10-30h)] [emotional] いらんアクション多い。スキップとか難易度さげてとかやってほしい、こういうゲームやる女苦手だよこういうの
    → Complains about too many unnecessary action sequences, wants skip options or difficulty reduction. States women who play these games aren't good at this type of gameplay.

--- story_content (1건) ---
  [regular(2-10h)] [emotional] 海に潜

In [32]:
# ── 일본어 company 카테고리 심층 분석 (넥슨 vs 가격 vs 기타) ──
ja_company = cls_ja[cls_ja['category'] == 'company']
if len(ja_company) > 0:
    print(f'=== 일본어 company 부정 ({len(ja_company)}건) ===')
    print(f'tone: {ja_company["tone"].value_counts().to_dict()}')
    print(f'sentiment: {ja_company["actual_sentiment"].value_counts().to_dict()}')
    print(f'\n--- 전체 리뷰 ---')
    for _, row in ja_company.iterrows():
        print(f'  [{row["tone"]}] [{row["actual_sentiment"]}] {row["review_text"][:300]}')
        print(f'    → {row["reason"]}')
        print()

=== 일본어 company 부정 (1건) ===
tone: {'emotional': 1}
sentiment: {'pure_negative': 1}

--- 전체 리뷰 ---
  [emotional] [pure_negative] 個人や少人数開発のインディーゲームが好きで、簡素な絵柄、単純明快で気軽なゲームを期待していましたが、胸焼けする程「やり込み要素」がメインです。魚の種類やレシピなど、尋常じゃない数が用意されてます。潜って魚を捕って寿司屋で売るというゲームシステムは簡単ですが、操作性も悪く６回ほど潜った辺りで「時間の無駄」と感じました。

インディーゲーム風なのは見た目だけで、大手メーカーが裏で入念にマーケティングし、お金の掛かる演出やゲームのボリュームを嵩増ししただけのクソゲーです。
    → Accuses game of being fake indie with big publisher marketing behind it ('クソゲー'), felt like waste of time after 6 dives, criticizes bloated content and poor controls. Strong negative sentiment about g



In [33]:
conn.close()
print('DB 연결 종료')

DB 연결 종료
